In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from scipy.sparse import hstack

# Функция для сохранения ответа без лишних переносов строк
def save_answer(filename, answer_str):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(str(answer_str).strip())
    print(f"Ответ для '{filename}' успешно сохранен: {answer_str}")

# 1. Загрузка данных
train_df = pd.read_csv('datasets/salary-train.csv')
test_df = pd.read_csv('datasets/salary-test-mini.csv')

# 2. Предобработка текстов (приведение к нижнему регистру и замена спецсимволов на пробелы)
# Используем регулярное выражение [^a-zA-Z0-9], чтобы оставить только буквы и цифры
def preprocess_text(series):
    return series.str.lower().replace('[^a-zA-Z0-9]', ' ', regex=True)

train_df['FullDescription'] = preprocess_text(train_df['FullDescription'])
test_df['FullDescription'] = preprocess_text(test_df['FullDescription'])

# 3. Применение TfidfVectorizer
# Оставляем слова, которые встречаются хотя бы в 5 объектах (min_df=5)
tfidf = TfidfVectorizer(min_df=5)
X_train_text = tfidf.fit_transform(train_df['FullDescription'])
X_test_text = tfidf.transform(test_df['FullDescription'])

# 4. Обработка пропусков в категориальных признаках
# Пишем без inplace=True, напрямую перезаписывая столбцы
train_df['LocationNormalized'] = train_df['LocationNormalized'].fillna('nan')
train_df['ContractTime'] = train_df['ContractTime'].fillna('nan')

test_df['LocationNormalized'] = test_df['LocationNormalized'].fillna('nan')
test_df['ContractTime'] = test_df['ContractTime'].fillna('nan')

# 5. Кодирование категориальных признаков через DictVectorizer (One-Hot Encoding)
# Переводим колонки в список словарей [{признак: значение}, ...]
enc = DictVectorizer()
train_cats = train_df[['LocationNormalized', 'ContractTime']].to_dict('records')
test_cats = test_df[['LocationNormalized', 'ContractTime']].to_dict('records')

X_train_cats = enc.fit_transform(train_cats)
X_test_cats = enc.transform(test_cats)

# 6. Объединение текстовых и категориальных признаков в единую разреженную матрицу
X_train = hstack([X_train_text, X_train_cats])
X_test = hstack([X_test_text, X_test_cats])

y_train = train_df['SalaryNormalized']

# 7. Обучение гребневой регрессии (Ridge) с alpha=1
model = Ridge(alpha=1.0, random_state=241)
model.fit(X_train, y_train)

# 8. Построение прогнозов для двух примеров из тест-мини файла
predictions = model.predict(X_test)

# Форматируем ответ: два числа через пробел, округляем до двух знаков
ans_str = f"{predictions[0]:.2f} {predictions[1]:.2f}"
save_answer('linreg_answer.txt', ans_str)

Ответ для 'linreg_answer.txt' успешно сохранен: 56572.23 37191.87
